# 4. Матричные разложения: SVD

В библиотеке `surprise` алгоритм `SVD` реализует разложение в стиле Funk SVD. Оценка предсказывается как

$$\hat r_{ui} = \mu + b_u + b_i + q_i^T p_u$$

Здесь $\mu$ обозначает глобальное среднее, $b_u$ и $b_i$ смещения пользователя и книги, а $p_u$ и $q_i$ их векторы скрытых признаков.
Параметры подбираются стохастическим градиентным спуском по известным оценкам с L2-регуляризацией.
Пропуски в матрице в обучении не участвуют, поэтому плотная матрица не нужна.

План: обучить модель на том же train, что и раньше, оценить RMSE на тесте, реализовать `get_recommendations`.

In [1]:
import numpy as np
import pandas as pd
from surprise import SVD, Dataset, Reader, accuracy
from surprise.model_selection import GridSearchCV, KFold

from utils import (
    N_BOOKS, N_USERS, SEED, TOP_K,
    get_eval_users, load_books, load_ratings, recommend_top_k, save_recs, split_ratings, to_sparse,
)

ratings = load_ratings()
books = load_books()
train, test = split_ratings(ratings)
eval_users = get_eval_users(test)
train_matrix = to_sparse(train)

print(f"train: {len(train)}, test: {len(test)}")

train: 784517, test: 194961


## Данные в формате surprise

Разбиение то же, что и в остальных ноутбуках (`split_ratings`): 20% оценок каждого пользователя в тесте.

In [2]:
reader = Reader(rating_scale=(1, 5))
columns = ["user_id", "book_id", "rating"]

train_data = Dataset.load_from_df(train[columns], reader)
trainset = train_data.build_full_trainset()
testset = list(test[columns].itertuples(index=False, name=None))

print(f"Пользователей: {trainset.n_users}, книг: {trainset.n_items}, оценок: {trainset.n_ratings}")

Пользователей: 53424, книг: 10000, оценок: 784517


## Подбор гиперпараметров

Небольшой перебор по сетке с кросс-валидацией на обучающей выборке (3 фолда). Тест в подборе не участвует.

In [3]:
%%time
param_grid = {
    "n_factors": [50, 100],
    "n_epochs": [20, 40],
    "reg_all": [0.02, 0.05, 0.1],
    "random_state": [SEED],
}
grid = GridSearchCV(
    SVD, param_grid, measures=["rmse"],
    cv=KFold(n_splits=3, random_state=SEED, shuffle=True), n_jobs=-1,
)
grid.fit(train_data)

CPU times: user 10.4 s, sys: 383 ms, total: 10.8 s
Wall time: 15.5 s


In [4]:
cv_results = pd.DataFrame(grid.cv_results)
cv_table = (
    cv_results[["param_n_factors", "param_n_epochs", "param_reg_all", "mean_test_rmse", "std_test_rmse"]]
    .rename(columns={
        "param_n_factors": "n_factors", "param_n_epochs": "n_epochs", "param_reg_all": "reg_all",
        "mean_test_rmse": "RMSE (CV)", "std_test_rmse": "std",
    })
    .sort_values("RMSE (CV)")
    .round(4)
    .reset_index(drop=True)
)
print("Лучшие параметры:", grid.best_params["rmse"])
cv_table

Лучшие параметры: {'n_factors': 50, 'n_epochs': 40, 'reg_all': 0.1, 'random_state': 42}


,n_factors,n_epochs,reg_all,RMSE (CV),std
0,50,40,0.10,0.8448,0.0010
1,100,40,0.10,0.8453,0.0008
2,50,40,0.05,0.8486,0.0010
3,50,20,0.10,0.8498,0.0009
4,50,20,0.05,0.8507,0.0009
5,100,40,0.05,0.8508,0.0006
6,100,20,0.10,0.8512,0.0008
7,50,20,0.02,0.8522,0.0009
8,100,20,0.05,0.8534,0.0007
9,50,40,0.02,0.8559,0.0009


Разница между конфигурациями небольшая. Сильнее всего влияет регуляризация: со значением по умолчанию (0.02)
модель при 40 эпохах начинает переобучаться, что ожидаемо при 15 оценках на пользователя в среднем.
Число факторов почти не играет роли. Лучшая конфигурация лежит на границе сетки (`reg_all = 0.1`, 40 эпох),
сетку можно было бы расширить, но речь идёт о третьем знаке RMSE, на выводы это не повлияет.

## Обучение и RMSE на тесте

In [5]:
%%time
algo = SVD(**grid.best_params["rmse"])
algo.fit(trainset)

predictions = algo.test(testset)
svd_rmse = accuracy.rmse(predictions, verbose=False)

CPU times: user 5.87 s, sys: 50.4 ms, total: 5.92 s
Wall time: 5.92 s


In [6]:
def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))


global_mean = train["rating"].mean()
book_means = train.groupby("book_id")["rating"].mean().reindex(test["book_id"]).fillna(global_mean).to_numpy()

rmse_table = pd.DataFrame(
    {
        "RMSE на тесте": [
            rmse(test["rating"], np.full(len(test), global_mean)),
            rmse(test["rating"], book_means),
            svd_rmse,
        ]
    },
    index=["Глобальное среднее", "Средняя оценка книги", "SVD"],
).round(4)
rmse_table

,RMSE на тесте
Глобальное среднее,0.9824
Средняя оценка книги,0.9458
SVD,0.8294


SVD ошибается в среднем на 0.83 балла. Это примерно на 16% лучше, чем предсказание глобальным средним, и лучше Item-based CF
из прошлого ноутбука (0.85 на том же тесте). Выигрыш небольшой. При такой короткой истории пользователя
значительную часть качества дают смещения $b_u$ и $b_i$, а не скрытые факторы.

In [7]:
errors = pd.DataFrame(
    [(p.r_ui, p.est) for p in predictions], columns=["rating", "predicted"]
)
errors.groupby("rating")["predicted"].agg(["mean", "std", "count"]).round(2)

,mean,std,count
rating,,,
1,3.25,0.67,3808
2,3.40,0.47,12577
3,3.60,0.41,49024
4,3.85,0.40,71207
5,4.22,0.44,58345


По разбивке видно, что предсказания стянуты к среднему. Для настоящих единиц и двоек
модель в среднем предсказывает больше 3, то есть низкие оценки она почти не угадывает. Их в данных слишком мало.

## Рекомендации для пользователя

`algo.predict(user, book)` считает оценку для одной пары. Чтобы получить top-N, нужно оценить все 10 000 книг,
поэтому достанем из модели матрицы факторов и смещения и посчитаем предсказания для всех книг сразу одним умножением.

В `surprise` внутренние индексы не совпадают с исходными id, поэтому сначала переставим строки в порядке `user_id` и `book_id`.

In [8]:
assert trainset.n_users == N_USERS and trainset.n_items == N_BOOKS  # все пользователи и книги есть в train

user_inner = np.array([trainset.to_inner_uid(uid) for uid in range(1, N_USERS + 1)])
book_inner = np.array([trainset.to_inner_iid(bid) for bid in range(1, N_BOOKS + 1)])

P, user_bias = algo.pu[user_inner], algo.bu[user_inner]
Q, book_bias = algo.qi[book_inner], algo.bi[book_inner]
mu = trainset.global_mean

print(f"Факторы пользователей: {P.shape}, факторы книг: {Q.shape}")


def svd_scores(user_idx):
    """Предсказанные оценки всех книг для набора пользователей (индексы с нуля)."""
    return P[user_idx] @ Q.T + user_bias[user_idx, None] + book_bias[None, :] + mu


# проверка, что формула совпадает с algo.predict
manual = svd_scores(np.array([0]))[0, :5]
library = [algo.predict(1, book_id, clip=False).est for book_id in range(1, 6)]
print("Совпадает с algo.predict:", np.allclose(manual, library))

Факторы пользователей: (53424, 50), факторы книг: (10000, 50)
Совпадает с algo.predict: True


In [9]:
def get_recommendations(user_id: int, N: int = 5) -> pd.DataFrame:
    """Top-N книг с наибольшим предсказанным рейтингом среди тех, что пользователь ещё не оценил."""
    recs = recommend_top_k(svd_scores, seen=train_matrix, user_ids=[user_id], k=N)[0]
    result = books.set_index("book_id").loc[recs, ["title", "authors"]].reset_index()
    result["predicted_rating"] = np.clip(svd_scores(np.array([user_id - 1]))[0, recs - 1], 1, 5).round(2)
    return result


user_id = int(train.groupby("user_id").size().loc[lambda s: s == 15].index[0])
history = train[train["user_id"] == user_id].merge(books[["book_id", "title"]], on="book_id")
print(f"Пользователь {user_id}, оценённые книги:")
print(history.sort_values("rating", ascending=False)[["rating", "title"]].to_string(index=False))

get_recommendations(user_id, N=5)

Пользователь 22, оценённые книги:
 rating                                                     title
      5        The Long Walk: The True Story of a Trek to Freedom
      4       The Walking Dead, Book One (The Walking Dead #1-12)
      4 Gates of Fire: An Epic Novel of the Battle of Thermopylae
      4          Day by Day Armageddon (Day by Day Armageddon,#1)
      4           The Dragon's Path (The Dagger and the Coin, #1)
      4    The Walking Dead, Book Three (The Walking Dead #25-36)
      4         Sin City, Vol. 3: The Big Fat Kill (Sin City, #3)
      3                           Empire of Ivory (Temeraire, #4)
      3           The Short Victorious War (Honor Harrington, #3)
      3                                            The Algebraist
      3               Black Sun Rising (The Coldfire Trilogy, #1)
      3                  Field of Dishonor (Honor Harrington, #4)
      3                Honor Among Enemies (Honor Harrington, #6)
      3                      Flag in Exile

,book_id,title,authors,predicted_rating
0,6590,The Authoritative Calvin and Hobbes: A Calvin ...,Bill Watterson,4.34
1,6920,The Indispensable Calvin and Hobbes,Bill Watterson,4.34
2,862,"Words of Radiance (The Stormlight Archive, #2)",Brandon Sanderson,4.33
3,5207,The Days Are Just Packed: A Calvin and Hobbes ...,Bill Watterson,4.32
4,7254,Homicidal Psycho Jungle Cat: A Calvin and Hobb...,Bill Watterson,4.31


In [10]:
get_recommendations(user_id=2, N=5)

,book_id,title,authors,predicted_rating
0,5207,The Days Are Just Packed: A Calvin and Hobbes ...,Bill Watterson,4.95
1,862,"Words of Radiance (The Stormlight Archive, #2)",Brandon Sanderson,4.91
2,6590,The Authoritative Calvin and Hobbes: A Calvin ...,Bill Watterson,4.90
3,7254,Homicidal Psycho Jungle Cat: A Calvin and Hobb...,Bill Watterson,4.89
4,6920,The Indispensable Calvin and Hobbes,Bill Watterson,4.88


Рекомендации разных пользователей во многом совпадают и состоят из книг с очень высоким средним рейтингом
(сборники "Кельвина и Хоббса", "Слова сияния" Сандерсона). С историей пользователя они связаны слабо.
Причина в формуле. При ранжировании по предсказанной оценке главную роль играет смещение книги $b_i$, одинаковое для всех,
а персональная часть $q_i^T p_u$ при короткой истории мала.

In [11]:
top_bias = np.argsort(-book_bias)[:5]
pd.DataFrame({
    "title": books.loc[top_bias, "title"].values,
    "book_bias": book_bias[top_bias].round(3),
})

,title,book_bias
0,The Days Are Just Packed: A Calvin and Hobbes ...,0.931
1,The Authoritative Calvin and Hobbes: A Calvin ...,0.913
2,"Words of Radiance (The Stormlight Archive, #2)",0.892
3,The Indispensable Calvin and Hobbes,0.882
4,The Complete Calvin and Hobbes,0.869


## Рекомендации для тестовых пользователей

Считаем top-10 для всех тестовых пользователей батчами (матрица предсказаний на всех сразу заняла бы около 3 ГБ)
и сохраняем для сравнения моделей.

In [12]:
%%time
recs_svd = recommend_top_k(svd_scores, seen=train_matrix, user_ids=eval_users, k=TOP_K)
save_recs("svd", eval_users, recs_svd)

print(f"Разных книг в рекомендациях: {len(np.unique(recs_svd))} из {N_BOOKS}")

Разных книг в рекомендациях: 105 из 10000
CPU times: user 1.64 s, sys: 47.5 ms, total: 1.69 s
Wall time: 1.62 s


## Итог

- SVD обучен на train, RMSE на тесте 0.83 против 0.98 у глобального среднего и 0.85 у Item-based CF.
- Обучение занимает секунды и линейно по числу оценок: $O(\text{nnz} \cdot f \cdot \text{эпохи})$, где $f$ это число факторов.
  Модель хранит только $(m + n) \cdot f$ чисел, поэтому масштабируется гораздо лучше матрицы сходств.
- `get_recommendations(user_id, N)` возвращает книги с наибольшей предсказанной оценкой. Уже по примерам видно,
  что такая выдача слабо персонализирована. Как это сказывается на метриках ранжирования, посмотрим в следующем ноутбуке.